# LangChain Expression Language (LCEL) with Ollama
This notebook provides hands-on, executable examples of the LCEL concepts covered in the [AI Engineering Visualized guide](https://amara-manikanta.github.io/ai-engineering-visualized/agents/langchain.html).

Ensure you have Ollama installed locally and have pulled the `llama3` model before running:
`ollama run llama3`

In [1]:
!pip install langchain-core langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.69
    Uninstalling langchain-core-0.3.69:
      Successfully uninstalled langchain-core-0.3.69
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-ollama][langchain-core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.4.9 which is incompatible.
langchain 0.3.26 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.4.9 which is incompatible.


In [8]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 6.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


## 1. Basic LCEL Chain (Pipe Operator)

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("Tell me a short, 1-sentence joke about {topic}")
model = OllamaLLM(model="llama3")
parser = StrOutputParser()

# Compose the chain using the pipe operator
chain = prompt | model | parser

# Invoke the chain
response = chain.invoke({"topic": "cats"})
print(response)

Why did the cat join a band? Because it wanted to be the purr-cussionist!


## 2. Streaming

In [5]:
for chunk in chain.stream({"topic": "bears"}):
    print(chunk, end="", flush=True)

Why did the bear go to the doctor? Because it had a grizzly cough!

## 3. Batching

In [6]:
responses = chain.batch([
    {"topic": "bears"}, 
    {"topic": "cats"}, 
    {"topic": "dogs"}
])

for i, resp in enumerate(responses):
    print(f"Joke {i+1}: {resp}")

Joke 1: Why did the bear go to the doctor? Because it had a grizzly cough!
Joke 2: Why did the cat join a band? Because it wanted to be the purr-cussionist!
Joke 3: Why did the dog go to the vet? Because it was feeling ruff!


## 4. Runnable Parallel

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def fake_retriever(query: str):
    return "Bears love eating honey and catching salmon."

parallel_setup = RunnableParallel(
    context=fake_retriever, 
    topic=RunnablePassthrough()
)

rag_prompt = PromptTemplate.from_template("Context: {context}\n\nTell me a joke about: {topic}")

rag_chain = parallel_setup | rag_prompt | model | parser

print(rag_chain.invoke("bears"))

Why did the bear go to the bees' party?

Because he was a bear-ly invited guest, but he just wanted to get his paws on some sweet honey!
